# Conservative numerical-reasoning experiment

**Hypothesis.** Explicit arithmetic over safely parsed, question-retrieved values may address a limitation of extractive selection. **Scope:** one arithmetic operation over two compatible values from the top three lexical evidence segments. The component abstains for ambiguity, incompatible metadata, unsafe parsing, or missing operands. Train fits retrieval vocabulary; dev is evaluated once; test is not loaded.

In [1]:
from __future__ import annotations

import json
import re
import tomllib
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from finevidence.data import build_structured_input, load_finqa_split
from finevidence.evaluation import ComparisonStatus, compare_display_answers, parse_display_answer
from finevidence.reasoning import parse_numeric_candidate, reason_over_candidates

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
config = tomllib.loads((PROJECT_ROOT / "configs" / "conservative_arithmetic.toml").read_text())
RAW_DIR = PROJECT_ROOT / config["dataset_root"]
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
train_records = load_finqa_split(RAW_DIR / f"{config['train_split']}.json")
dev_records = load_finqa_split(RAW_DIR / f"{config['evaluation_split']}.json")
print({"train": len(train_records), "dev": len(dev_records), "test_loaded": False})

{'train': 6251, 'dev': 883, 'test_loaded': False}


In [2]:
def segments(record):
    source = build_structured_input(record)
    header = [str(cell) for cell in source["table"][0]]
    output = []
    for index, row in enumerate(source["table"][1:], start=1):
        output.append(
            {
                "key": f"table_{index}",
                "type": "table",
                "row": row,
                "header": header,
                "text": " | ".join(map(str, row)),
            }
        )
    for index, text in enumerate(source["pre_text"] + source["post_text"]):
        output.append({"key": f"text_{index}", "type": "text", "text": text})
    return output


def candidates(segment):
    output = []
    if segment["type"] == "table":
        for _column_index, raw in enumerate(segment["row"][1:], start=1):
            output.append(
                parse_numeric_candidate(
                    str(raw), source_id=segment["key"], context=segment["text"], period=None
                )
            )
    else:
        for match in re.finditer(
            r"(?<![\w.])[-+]?(?:\d+(?:,\d{3})*(?:\.\d+)?|\.\d+)%?(?![\w.])", segment["text"]
        ):
            output.append(
                parse_numeric_candidate(
                    match.group(), source_id=segment["key"], context=segment["text"]
                )
            )
    return output


train_documents = [
    text
    for record in train_records
    for text in [
        build_structured_input(record)["question"],
        *(item["text"] for item in segments(record)),
    ]
]
vectorizer = TfidfVectorizer(
    lowercase=True, ngram_range=(1, 2), max_features=config["max_features"]
)
vectorizer.fit(train_documents)
print({"train_documents": len(train_documents), "vocabulary": len(vectorizer.vocabulary_)})

{'train_documents': 191354, 'vocabulary': 50000}


In [3]:
rows = []
for record in dev_records:
    source = build_structured_input(record)
    evidence = segments(record)
    q = vectorizer.transform([source["question"]])
    scores = (q @ vectorizer.transform([item["text"] for item in evidence]).T).toarray().ravel()
    selected = [
        evidence[index] for index in scores.argsort()[::-1][: config["top_evidence_segments"]]
    ]
    trace = reason_over_candidates(
        source["question"], [candidate for item in selected for candidate in candidates(item)]
    )
    prediction = str(trace.result) if trace.answered else None
    comparison = compare_display_answers(prediction, record["qa"]["answer"])
    parsed = parse_display_answer(record["qa"]["answer"])
    rows.append(
        {
            "id": record["id"],
            "question": source["question"],
            "prediction": prediction,
            "reference": record["qa"]["answer"],
            "type": "empty"
            if parsed.kind == "empty"
            else "numeric_percent"
            if parsed.kind == "numeric" and parsed.is_percent
            else parsed.kind,
            "status": comparison.status.value,
            "answered": trace.answered,
            "intent": trace.hypothesis.intent.value,
            "operation": trace.hypothesis.operation.value if trace.hypothesis.operation else None,
            "abstention": trace.abstention_reason,
            "operands": [(operand.raw_value, operand.source_id) for operand in trace.operands],
            "evidence": trace.evidence_references,
            "retrieved": [item["key"] for item in selected],
        }
    )
frame = pd.DataFrame(rows)
eligible = frame[frame["status"].ne(ComparisonStatus.UNAVAILABLE_REFERENCE.value)]
metrics = pd.DataFrame(
    {
        "dev_records": [len(frame)],
        "denominator": [len(eligible)],
        "empty_references": [len(frame) - len(eligible)],
        "correct": [eligible["status"].eq(ComparisonStatus.CORRECT.value).sum()],
        "answered": [frame["answered"].sum()],
    }
)
metrics["accuracy"] = metrics["correct"] / metrics["denominator"]
metrics["answered_accuracy"] = (
    eligible[eligible["answered"]]["status"].eq(ComparisonStatus.CORRECT.value).mean()
)
display(metrics)
reference_path = OUTPUT_DIR / "lexical_value_baseline_dev_predictions.jsonl"
reference_rows = [json.loads(line) for line in reference_path.open(encoding="utf-8")]
reference_frame = pd.DataFrame(reference_rows)
reference_eligible = reference_frame[reference_frame["metric_eligible"]]
comparison = pd.DataFrame(
    {
        "system": ["lexical-value-selection reference", "conservative arithmetic"],
        "correct": [
            reference_eligible["comparison_status"].eq(ComparisonStatus.CORRECT.value).sum(),
            metrics.loc[0, "correct"],
        ],
        "denominator": [len(reference_eligible), metrics.loc[0, "denominator"]],
    }
)
comparison["accuracy"] = comparison["correct"] / comparison["denominator"]
display(comparison)
display(frame["intent"].value_counts().rename("records").to_frame())
display(frame["abstention"].fillna("answered").value_counts().rename("records").to_frame())
display(
    eligible.assign(correct=eligible["status"].eq(ComparisonStatus.CORRECT.value))
    .groupby("type")
    .agg(denominator=("correct", "size"), correct=("correct", "sum"))
    .assign(accuracy=lambda x: x.correct / x.denominator)
)
display(
    frame[
        [
            "id",
            "question",
            "prediction",
            "reference",
            "operands",
            "operation",
            "evidence",
            "status",
            "abstention",
        ]
    ].head(10)
)

,dev_records,denominator,empty_references,correct,answered,accuracy,answered_accuracy
0,883,871,12,32,293,0.036739,0.110345


,system,correct,denominator,accuracy
0,lexical-value-selection reference,3,871,0.003444
1,conservative arithmetic,32,871,0.036739


,records
intent,
ambiguous,520
additive,130
subtractive,121
divisive,106
comparison,5
multiplicative,1


,records
abstention,
unsupported or ambiguous question intent,525
answered,293
mismatched unit or period metadata,38
insufficient safely parsed operands,27


,denominator,correct,accuracy
type,,,
numeric,356,32,0.089888
numeric_percent,492,0,0.000000
text,23,0,0.000000


,id,question,prediction,reference,operands,operation,evidence,status,abstention
0,V/2008/page_17.pdf-1,what is the average payment volume per transac...,NaN,127.40,[],NaN,(),unsupported_prediction,unsupported or ambiguous question intent
1,C/2017/page_328.pdf-1,what was the percentage cumulative total retur...,NaN,93.5%,[],NaN,(),unsupported_prediction,unsupported or ambiguous question intent
2,DVN/2007/page_58.pdf-2,what percentage of the total oil and gas mmboe...,NaN,24.69%,[],NaN,(),unsupported_prediction,unsupported or ambiguous question intent
3,ETR/2011/page_341.pdf-3,in 2010 what was the net change in net revenue...,18.6,18.6,"[($ 555.3, table_4), ($ 536.7, table_1)]",subtract,"(table_4, table_1)",correct,NaN
4,ETR/2004/page_213.pdf-2,what are the deferred fuel cost revisions as a...,NaN,60.3%,[],NaN,(),unsupported_prediction,unsupported or ambiguous question intent
5,PM/2017/page_38.pdf-1,what was the change in millions of operating i...,688,688,"[(11503, table_3), (10815, table_3)]",subtract,"(table_3, table_3)",correct,NaN
6,JPM/2008/page_85.pdf-3,what was jpmorgan chase & co's common equity t...,1.533635318774930701102021500,10.94%,"[($ 136104, table_1), ($ 88746, table_1)]",divide,"(table_1, table_1)",incorrect,NaN
7,PNC/2013/page_207.pdf-1,"at the end of 2014 , the notional value of der...",30.44322960470984020185029437,3044%,"[($ 36197, table_1), ($ 1189, table_1)]",divide,"(table_1, table_1)",incorrect,NaN
8,CME/2017/page_97.pdf-5,what was the increase in class a common stock ...,995,995,"[(339235, table_2), (338240, table_2)]",subtract,"(table_2, table_2)",correct,NaN
9,PNC/2013/page_62.pdf-2,in millions what was total residential mortgag...,3576,3576,"[(1356, table_6), (2220, table_6)]",add,"(table_6, table_6)",correct,NaN


In [4]:
# Post-prediction only: diagnostics use published annotations after outputs are fixed.
post_hoc = []
for row, record in zip(rows, dev_records, strict=True):
    operations = re.findall(r"([a-z_]+)\(", record["qa"]["program"])
    post_hoc.append(
        {
            "multi_step_program": len(operations) > 1,
            "multiple_gold_sources": len(record["qa"]["gold_inds"]) > 1,
            "temporal_cue": bool(
                re.search(
                    r"\b(?:from|to|between|year|quarter|ended|versus|compared)\b",
                    row["question"],
                    re.I,
                )
            ),
            "unit_conversion_cue": bool(
                re.search(
                    r"\b(?:percent|percentage|million|billion|thousand|per)\b",
                    row["question"],
                    re.I,
                )
            ),
        }
    )
display(pd.DataFrame(post_hoc).sum().rename("dev_records").to_frame())
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
path = OUTPUT_DIR / "conservative_arithmetic_dev_predictions.jsonl"
with path.open("w", encoding="utf-8") as handle:
    for row in rows:
        handle.write(json.dumps(row) + "\n")
print(path.relative_to(PROJECT_ROOT))

,dev_records
multi_step_program,360
multiple_gold_sources,467
temporal_cue,517
unit_conversion_cue,398


data/processed/conservative_arithmetic_dev_predictions.jsonl


## Limits

This is an explicit arithmetic component, not a general FinQA executor. It never reads gold programs or labels at prediction time, does not convert percentages/scales/currencies, and abstains whenever evidence alignment is not safe. Gold-evidence retrieval cannot establish that all operands required for a calculation were retrieved.